In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
pip install langchain-groq langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [ ]:

import os
os.environ["GOOGLE_API_KEY"] = "google api key"
os.environ["GROQ_API_KEY"]="groq api key"

In [5]:
import os
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


**LOAD DATA INGESTION**

In [6]:
video_id = "4dBWH8FmP4E"
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id)

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)


except TranscriptsDisabled:
    print("No captions available for this video.")

[Music] Hi everyone. Today I'm here to clear up one of the biggest points of confusion in tech right now. That is what's the difference between generative AI, agentic AI, and AI agents. We hear these terms everywhere, but what do they actually mean? People often mix them because they sound similar but in reality each one works differently, has a different role and comes with unique strength and weaknesses. So today I want to walk you through these three terms in simplest language possible. We will look at what they are, how they work, their internal model structures and what makes them different from each other. To make this fun, imagine three different assistant. One is a creative writer who can instantly craft poems, code or even pictures. The another one is a robot worker that follows rules and does not task again and again perfectly. And the third one is a butler with a brain. It can plan, decide, use tools and even bring in other helpers to flex job. That's the big picture. Now le

# **Step 1b - Indexing (Text Splitting)**





In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [8]:
len(chunks)

49

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

**\# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store**




In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
vector_store = FAISS.from_documents(chunks, embeddings)


In [12]:
vector_store.index_to_docstore_id

{0: 'f76c141b-8bbf-436e-8059-79b5f6a6067f',
 1: 'e12f8926-e760-4301-85f1-8dea36ceeaeb',
 2: '19f5d16a-03e1-4152-a552-6a309c9aeadc',
 3: '02500642-5cf7-422a-a0b6-5362efa9e3cd',
 4: 'd03a6449-5b62-4593-849b-be43aeb44175',
 5: '3beae3a7-739d-4499-a75d-9100ef5a0a1b',
 6: 'bcf968a3-2c7d-4b5c-832e-da855f97afae',
 7: '4155b61b-f3c8-46fe-bf95-b96dabb9d8f1',
 8: 'ee09b2a8-bed5-4684-b8cb-5f9580d40b24',
 9: 'f6259901-214a-4e17-9f2e-9ebb9c7e662c',
 10: 'bbc2becc-40fe-47e5-a35f-f8fb07d55a65',
 11: '75037931-18f0-433f-b811-84efcd79062f',
 12: '61c2c896-2bd1-4945-9ce4-2fdf1c0383cd',
 13: '92417cec-556d-489b-a399-9e43cb278ba0',
 14: 'f98359f5-24bd-43d4-a034-8ea15a09fd00',
 15: '8494d3f7-7797-4b2d-b3de-5e5c01aa7920',
 16: 'd0df420b-bc9b-43be-827a-94b5b4e90b46',
 17: 'f2371537-d153-4b6b-9ace-122bf5490502',
 18: 'e984e1fb-10b7-4843-b178-db561250b468',
 19: 'eb1b9dc1-a68c-4d5f-9eb6-440c6fc801dd',
 20: '9e180de8-1f00-4ac9-87ab-9967db0772a8',
 21: 'bc3615f7-b93d-4968-b1c7-51342d559066',
 22: '71dc19a6-3ce5-

In [13]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ded89da1e20>, search_kwargs={'k': 4})

In [14]:
retriever.invoke('What is genrative ai')


[Document(id='bbc2becc-40fe-47e5-a35f-f8fb07d55a65', metadata={}, page_content="images, code, music and even video. You can think of chart GBT mid journey or daily. You give the prompt and then they create an output. If I say write me a bedtime story about space traveling cat, it will instantly write one. That's generative AI in action. So let's talk about how it works. That is the model structure. Now the heart of generative AI is something called large language model LLM. This is like the brain of the system. So it's trained on billion of words from books, Wikipedia and"),
 Document(id='75037931-18f0-433f-b811-84efcd79062f', metadata={}, page_content="the model structure. Now the heart of generative AI is something called large language model LLM. This is like the brain of the system. So it's trained on billion of words from books, Wikipedia and online articles. So it understand how languages work. The model uses something called transformer architecture which basically breaks down t

# Step 3 Augmentation

In [15]:



llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1,
    max_tokens=1024,
    top_p=1,
    streaming=True
)



/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [16]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [17]:
question          = "diference between agentic ai and genrative ai"
retrieved_docs    = retriever.invoke(question)

In [18]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"comparison. We've got three levels of AI here. Generative AI, AI agents, and agentic AI. So, first let's talk about generative AI. You can think of this as a creative brain and its core it's powered only by a large language model or LLM. It can generate content like writing a story, creating an image or drafting an e. But that's just about it. No memory, no external tools, just pure content creation. Autonomy here is bit low because it only responds to your prompt and nothing more. Next, we're\n\ntask on its own. Of course, each comes with its own cautions. But with generative AI, you should always fast check because it can make mistakes. AI agents work well, but they have a limited scope and it needs regular rule updates. An agentic AI is powerful, but it must have strong guard rails or else it might go off track when making decisions. So now that you know the difference, generative AI is like a creative writer capable of producing new text, images, or ideas. AI agents act as reliabl

In [19]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [20]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      comparison. We've got three levels of AI here. Generative AI, AI agents, and agentic AI. So, first let's talk about generative AI. You can think of this as a creative brain and its core it's powered only by a large language model or LLM. It can generate content like writing a story, creating an image or drafting an e. But that's just about it. No memory, no external tools, just pure content creation. Autonomy here is bit low because it only responds to your prompt and nothing more. Next, we're\n\ntask on its own. Of course, each comes with its own cautions. But with generative AI, you should always fast check because it can make mistakes. AI agents work well, but they have a limited scope and it needs regular rule updates. An agentic AI is powerful, but it must have strong guard rails or else it mi

# Step 4 - Generation

In [21]:
answer = llm.invoke(final_prompt)
print(answer.content)

The main difference between Agentic AI and Generative AI is their level of autonomy and capabilities. 

Generative AI is like a creative brain, powered by a large language model, and its core function is to generate content such as writing, images, or ideas. It has low autonomy, only responding to prompts, and lacks memory and external tools.

Agentic AI, on the other hand, is an autonomous orchestrator that not only completes tasks but also reasons, plans, and coordinates multiple steps on its own. It has higher autonomy and can make decisions, but requires strong guard rails to prevent it from going off track.


# **Step-5 Chains**

In [22]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text


In [23]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [24]:
parser = StrOutputParser()

In [25]:
main_chain = parallel_chain | prompt | llm | parser

In [26]:
main_chain.invoke('Can you summarize the video')

'The video discusses different levels of AI, including generative AI and agentic AI. Agentic AI is described as having limited but real autonomy, able to complete tasks and make decisions on its own, such as booking a flight and finding the cheapest option. It works best with clear and simple tasks, but is not great at complex multi-step reasoning. The video also mentions that agentic AI can provide entire codes and summaries, going beyond just generating content.'